## 1. Objective

This notebook prepares the Telco Customer Churn dataset for machine learning. It includes:

- Cleaning and transforming raw features
- Encoding categorical variables
- Scaling numerical features
- Engineering new features (e.g., tenure buckets)
- Saving a clean, ready-to-model dataset


2. Load Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/telco_clean_base.csv")
print("✅ Dataset loaded:", df.shape)
df.head()

✅ Dataset loaded: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


3. Handle Missing Values & Data Types

In [2]:
# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop rows with missing TotalCharges
missing = df['TotalCharges'].isna().sum()
print(f"🕳️ Missing TotalCharges: {missing}")
df.dropna(subset=['TotalCharges'], inplace=True)

🕳️ Missing TotalCharges: 11



Some customers had blank `TotalCharges` (likely due to zero tenure). These rows were dropped to ensure numeric consistency.


4. Encode Categorical Variables

In [3]:
# Drop customerID (not predictive)
df.drop(columns='customerID', inplace=True)

# Binary encoding
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# SeniorCitizen is already 0/1
# Gender: optional — encode if useful
df['gender'] = df['gender'].map({'Female': 1, 'Male': 0})

# One-hot encode remaining categoricals
df = pd.get_dummies(df, drop_first=True)
print("✅ Encoded dataset shape:", df.shape)


✅ Encoded dataset shape: (7032, 31)



- Binary features were mapped to 0/1 for simplicity.  
- Multiclass features (e.g. `Contract`, `InternetService`) were one-hot encoded to avoid ordinal assumptions.  
- `customerID` was dropped as it carries no predictive value.


5. Feature Engineering — Tenure Buckets

In [4]:
def tenure_bucket(tenure):
    if tenure <= 12:
        return '0–12'
    elif tenure <= 24:
        return '13–24'
    elif tenure <= 48:
        return '25–48'
    elif tenure <= 60:
        return '49–60'
    else:
        return '61+'

df['tenure_group'] = df['tenure'].apply(tenure_bucket)
df = pd.get_dummies(df, columns=['tenure_group'], drop_first=True)



Binning `tenure` helps capture non-linear effects and customer lifecycle stages. This can improve model interpretability and performance.


6. Scale Numerical Features

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[num_cols] = scaler.fit_transform(df[num_cols])

print("✅ Scaled numerical features.")
df[num_cols].describe()


✅ Scaled numerical features.


,tenure,MonthlyCharges,TotalCharges
count,7.032000e+03,7.032000e+03,7.032000e+03
mean,-1.126643e-16,6.062651e-17,-1.119064e-16
std,1.000071e+00,1.000071e+00,1.000071e+00
min,-1.280248e+00,-1.547283e+00,-9.990692e-01
25%,-9.542963e-01,-9.709769e-01,-8.302488e-01
50%,-1.394171e-01,1.845440e-01,-3.908151e-01
75%,9.199259e-01,8.331482e-01,6.668271e-01
max,1.612573e+00,1.793381e+00,2.824261e+00



Standardizing numerical features ensures that models like Logistic Regression and Gradient Boosting treat all features on the same scale. This is especially important for distance-based algorithms or regularization.


7. Save Final Dataset

In [6]:
df.to_csv("../data/processed/telco_final_preprocessed.csv", index=False)
print("💾 Final dataset saved.")


💾 Final dataset saved.


## Summary

- Cleaned and encoded all categorical variables
- Scaled numerical features using StandardScaler
- Engineered tenure buckets to capture customer lifecycle
- Saved the final dataset for modeling

 
